
# Advanced Evaluation Metrics

In "Simple Linear Regression," we introduced standard metrics like RMSE and $R^2$. However, real-world problems often require more nuance.

For example, $R^2$ has a flaw: it **always** increases on the *training set* when you add features, even if they are useless. And RMSE can be dangerously sensitive to outliers. In this module, we explore advanced metrics that protect us from these pitfalls.

## MAE vs. RMSE: The Penalty Game

Both measure the "average error," but they treat mistakes differently.

-   **MAE (L1 Norm)**: Treats all errors linearly. Being off by 10 is exactly twice as bad as being off by 5.
-   **RMSE (L2 Norm)**: Squares the errors. Being off by 10 is **four times** as bad as being off by 5.

**Rule of Thumb**:

-   Use **MAE** if you want a robust metric that reflects "typical" performance.
-   Use **RMSE** if large errors are unacceptable (e.g., in medical dosages or safety systems).

## Adjusted $R^2$: The "BS Detector"

Standard $R^2$ has a major weakness: if you add a feature filled with random noise, $R^2$ will stay the same or slightly increase. It never decreases, at least on the *training set*, though it may do so on the *test set*. This incentivizes adding infinite garbage features (Overfitting).

**Adjusted $R^2$** penalizes the model for adding useless features. $$R^2_{adj} = 1 - (1-R^2) \frac{n-1}{n-p-1}$$ Where $n$ is samples and $p$ is features.

-   If you add a feature and the model doesn't improve significantly, Adjusted $R^2$ **drops**.

## RMSLE: The Log Error

When predicting prices (like houses or stocks), an error of $50k is huge on a $100k house, but negligible on a $10M house. **Root Mean Squared Logarithmic Error (RMSLE)** cares about the **ratio**, not the absolute difference.

### Why RMSLE Penalizes Under-Prediction More

RMSLE computes $\log(pred + 1) - \log(actual + 1)$. Due to the asymmetry of logarithms:

-   Predicting **2x too high**: $\log(2) \approx 0.69$
-   Predicting **2x too low** (half): $\log(0.5) \approx -0.69$ → same magnitude

But for the **same absolute error**, under-prediction hurts more:

In [ ]:
import numpy as np

actual = 100
over_pred = 150   # +50 error
under_pred = 50   # -50 error

error_over = np.log1p(over_pred) - np.log1p(actual)
error_under = np.log1p(under_pred) - np.log1p(actual)

print(f"Over-predict by 50:  log error = {error_over:.4f}")
print(f"Under-predict by 50: log error = {error_under:.4f}")
print(f"Under-prediction penalty is {abs(error_under)/abs(error_over):.2f}x larger")

### Outlier Resistance (Compared to RMSE)

RMSLE compresses large values via the logarithm, reducing the influence of extreme predictions.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_squared_log_error

y_true = np.array([100, 100, 100, 100])
y_pred_normal = np.array([110, 90, 105, 95])    # Typical errors
y_pred_outlier = np.array([110, 90, 105, 500])  # One massive outlier

# RMSE comparison
rmse_normal = np.sqrt(mean_squared_error(y_true, y_pred_normal))
rmse_outlier = np.sqrt(mean_squared_error(y_true, y_pred_outlier))

# RMSLE comparison
rmsle_normal = np.sqrt(mean_squared_log_error(y_true, y_pred_normal))
rmsle_outlier = np.sqrt(mean_squared_log_error(y_true, y_pred_outlier))

print(f"RMSE  (normal): {rmse_normal:.2f}  → (with outlier): {rmse_outlier:.2f}  ({rmse_outlier/rmse_normal:.1f}x increase)")
print(f"RMSLE (normal): {rmsle_normal:.4f} → (with outlier): {rmsle_outlier:.4f} ({rmsle_outlier/rmsle_normal:.1f}x increase)")

**Observation**: RMSE explodes with the outlier; RMSLE increases but remains more stable.

**Caveat**: RMSLE fails if predictions are near zero (log blows up). Always ensure positive predictions.

## Practical Demonstration

We will compare a "Good Model" vs. a "Bloated Model" (one with random noise features) to see Adjusted $R^2$ in action.

### Setup Data with "Noise Features"

We add 10 columns of pure random noise to the California Housing dataset.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd

# Load Clean Data
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

# Generate 10 columns of random noise
np.random.seed(42)
noise = np.random.normal(0, 1, (X.shape[0], 10))
df_noise = pd.DataFrame(noise, columns=[f'Noise_{i}' for i in range(10)])

# Create "Bloated" dataset
X_bloated = pd.concat([X, df_noise], axis=1)

print(f"Original Features: {X.shape[1]}")
print(f"Bloated Features:  {X_bloated.shape[1]}")

### Compare $R^2$ vs Adjusted $R^2$

We define a helper function to calculate Adjusted $R^2$ since scikit-learn doesn't have it built-in. Then we compare the **Clean Model** vs the **Bloated Model** side-by-side.

In [ ]:
def adjusted_r2(r2, n, p):
    """ Calculates Adjusted R-squared.
    n: number of samples
    p: number of features
    """
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

# Split BOTH datasets with same random state for fair comparison
X_train_clean, X_test_clean, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
X_train_bloated, X_test_bloated, _, _ = train_test_split(
    X_bloated, y, test_size=0.2, random_state=42)

# Train Clean Model
model_clean = LinearRegression()
model_clean.fit(X_train_clean, y_train)
y_pred_clean = model_clean.predict(X_test_clean)

# Train Bloated Model
model_bloated = LinearRegression()
model_bloated.fit(X_train_bloated, y_train)
y_pred_bloated = model_bloated.predict(X_test_bloated)

# Calculate Metrics for Clean Model
n = X_test_clean.shape[0]
r2_clean = r2_score(y_test, y_pred_clean)
adj_r2_clean = adjusted_r2(r2_clean, n, X_test_clean.shape[1])

# Calculate Metrics for Bloated Model
r2_bloated = r2_score(y_test, y_pred_bloated)
adj_r2_bloated = adjusted_r2(r2_bloated, n, X_test_bloated.shape[1])

print("=" * 45)
print(f"{'Metric':<15} {'Clean (8 feat)':<15} {'Bloated (18 feat)':<15}")
print("=" * 45)
print(f"{'Standard R²':<15} {r2_clean:<15.5f} {r2_bloated:<15.5f}")
print(f"{'Adjusted R²':<15} {adj_r2_clean:<15.5f} {adj_r2_bloated:<15.5f}")
print("=" * 45)

**Observation**: Standard $R^2$ may look similar (or even slightly higher for the bloated model on training data), but Adjusted $R^2$ reveals the truth: the noise features add no value. As $p$ approaches $n$, this gap widens dramatically, though it is quite small here.

### RMSLE (Handling Skew)

House prices vary wildly. Let's see how RMSLE handles the error compared to RMSE. **Note**: RMSLE requires positive targets.

In [ ]:
from sklearn.metrics import mean_squared_log_error

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred_clean))

# Calculate RMSLE
# (California housing target is in 100k, so values are small, e.g., 2.5. 
# This metric is usually for raw prices like 250,000, but works here too).
# We ensure no negative predictions for RMSLE (clip at 0)
y_pred_safe = np.maximum(y_pred_clean, 0)
rmsle = np.sqrt(mean_squared_log_error(y_test, y_pred_safe))

print(f"RMSE:  {rmse:.4f} (Absolute Error)")
print(f"RMSLE: {rmsle:.4f} (Relative/Ratio Error)")

## Exercises

### The "Kitchen Sink" Experiment

Add 100, then 500 noise features to the dataset.

1.  Train a model for each.
2.  Plot $R^2$ vs Adjusted $R^2$.
3.  Watch them diverge. $R^2$ will decrease much slower than Adjusted $R^2$, which will tank.

### Prediction Error Plot

A powerful visual metric is the "Prediction Error Plot".

-   Plot `y_true` on the x-axis.
-   Plot `y_pred` on the y-axis.
-   Draw a 45-degree identity line.

Points below the line are **Underestimations**; points above are **Overestimations**.

## Summary

1.  **Context Matters**: Don't use RMSE just because everyone else does. If outliers are rare anomalies you want to ignore, use MAE.
2.  **Model Selection**: When comparing models with different numbers of features, **always use Adjusted $R^2$**, never standard $R^2$.
3.  **Visualization**: A Prediction Error Plot tells you more in 1 second than a single number ever could.